# US Data Preparation
Convert the original us_data_prep.py into a notebook. Outputs are saved with start-of-month index strings in `YYYY-MM-01` format.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import matplotlib.pyplot as plt
from dateutil.relativedelta import relativedelta
from statsmodels.tsa.stattools import adfuller

# Parameters
desired_start_date_of_samples = datetime.strptime('1961-01-01', '%Y-%m-%d') # earliest date for excess bond premium
last_date = datetime.strptime('2024-12-01', '%Y-%m-%d')
last_date_for_save = last_date.strftime('%Y-%m')
HORIZON_IN_QUARTERS = 4
DATA_DIR = '/home/rproner/Documents/Data/'

In [2]:
# --- Load FRED monthly file and build indices ---
fred_y = pd.read_csv(os.path.join(DATA_DIR, '2025-10-MD.csv')).iloc[1:]
fred_y['date'] = pd.date_range('1959-01-01', '2025-09-01', freq='MS')
fred_y.drop(columns=[c for c in ['sasdate'] if c in fred_y.columns], inplace=True, errors='ignore')
fred_y.set_index('date', inplace=True)


## Targets

In [3]:
# --- Targets ---
infl_t = np.log(fred_y['CPIAUCSL']) - np.log(fred_y['CPIAUCSL'].shift(12))
ip_t = np.log(fred_y['INDPRO']) - np.log(fred_y['INDPRO'].shift(12))
u_t = fred_y['UNRATE'] - fred_y['UNRATE'].shift(12)
lu_t = np.log(fred_y['UNRATE']) - np.log(fred_y['UNRATE'].shift(12))
rl_t = np.log(fred_y['REALLN']) - np.log(fred_y['REALLN'].shift(12))

targets = pd.DataFrame({
    'Infl_yoy': infl_t,
    'IP_yoy': ip_t,
    'Unrate_yoy': lu_t
    # 'RealLoans_yoy': rl_t
})


In [4]:
for col in targets.columns:
    result = adfuller(targets[col].dropna())
    print(f'ADF Statistic for {col}: {result[0]}')
    print(f'p-value: {result[1]}')
    for key, value in result[4].items():
        print(f'Critical Value {key}: {value}')
    print('---')

ADF Statistic for Infl_yoy: -3.256130613325994
p-value: 0.016957000730760844
Critical Value 1%: -3.438837902109151
Critical Value 5%: -2.8652862410999114
Critical Value 10%: -2.568764869203001
---
ADF Statistic for IP_yoy: -5.859935905016479
p-value: 3.429284173041936e-07
Critical Value 1%: -3.438882201132452
Critical Value 5%: -2.865305765357574
Critical Value 10%: -2.568775270215655
---
ADF Statistic for Unrate_yoy: -5.618204433273859
p-value: 1.1612396959571547e-06
Critical Value 1%: -3.4388268991356936
Critical Value 5%: -2.8652813916285518
Critical Value 10%: -2.5687622857867782
---


## Predictors

In [5]:
# AR(1) predictors 
ar1_predictors = pd.DataFrame({
    'Infl_yoy_t-1': infl_t.shift(3*HORIZON_IN_QUARTERS + 1), # Extra lag for announcement delays.
    'IP_yoy_t-1': ip_t.shift(3*HORIZON_IN_QUARTERS + 1),
    'Unrate_yoy_t-1': lu_t.shift(3*HORIZON_IN_QUARTERS + 1)
})
ar1_predictors.head(30)

,Infl_yoy_t-1,IP_yoy_t-1,Unrate_yoy_t-1
date,,,
1959-01-01,NaN,NaN,NaN
1959-02-01,NaN,NaN,NaN
1959-03-01,NaN,NaN,NaN
1959-04-01,NaN,NaN,NaN
1959-05-01,NaN,NaN,NaN
1959-06-01,NaN,NaN,NaN
1959-07-01,NaN,NaN,NaN
1959-08-01,NaN,NaN,NaN
1959-09-01,NaN,NaN,NaN


In [6]:
# --- Features: factors and nfci ---
fred = pd.read_csv(os.path.join(DATA_DIR, '2025-10-MD-transformed.csv'), index_col=0, parse_dates=True)
factors = pd.read_csv(os.path.join(DATA_DIR, '[usa]_[all_factors]_[monthly]_[vw_cap]_24-12.csv'))
oap = pd.read_csv(os.path.join(DATA_DIR, 'PredictorLSretWide.csv'), index_col=0)
nfci = pd.read_csv(os.path.join(DATA_DIR, 'nfci_monthly.csv'), index_col=0, parse_dates=True)

for df in (fred, factors, nfci):
    print(df.head(5))

                 RPI   W875RX1  DPCERA3M086SBEA  CMRMTSPLx   RETAILx  \
sasdate                                                                
1959-01-01       NaN       NaN              NaN        NaN       NaN   
1959-02-01  0.003877  0.003621         0.010349   0.007336  0.007310   
1959-03-01  0.006457  0.007325         0.009404  -0.003374  0.008321   
1959-04-01  0.006510  0.007029        -0.003622   0.019915  0.000616   
1959-05-01  0.005796  0.006618         0.012043   0.006839  0.007803   

              INDPRO   IPFPNSS   IPFINAL   IPCONGD  IPDCONGD  ...  \
sasdate                                                       ...   
1959-01-01       NaN       NaN       NaN       NaN       NaN  ...   
1959-02-01  0.019395  0.013405  0.008628  0.007309  0.005232  ...   
1959-03-01  0.014300  0.006036  0.004896  0.000000  0.019397  ...   
1959-04-01  0.021080  0.014339  0.014545  0.015652  0.006379  ...   
1959-05-01  0.014954  0.008267  0.009580  0.004766  0.020152  ...   

           

In [7]:
# Convert factor data to wide (pivot)
factors = factors[['name', 'date', 'ret']]
factors = pd.pivot_table(factors, values='ret', index='date', columns='name')
factors.index = pd.to_datetime(factors.index).to_period('M').to_timestamp()

oap.index = pd.to_datetime(oap.index).to_period('M').to_timestamp()
oap.head()

,AM,AOP,AbnormalAccruals,Accruals,AccrualsBM,Activism1,Activism2,AdExp,AgeIPO,AnalystRevision,...,retConglomerate,roaq,sfe,sinAlgo,skew1,std_turn,tang,zerotrade12M,zerotrade1M,zerotrade6M
date,,,,,,,,,,,,,,,,,,,,,
1926-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-13.686396,NaN,NaN,NaN,NaN,NaN,NaN
1926-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-5.135249,NaN,NaN,NaN,NaN,NaN,NaN
1926-03-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-4.832504,NaN,NaN,NaN,NaN,NaN,NaN
1926-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-4.440908,NaN,NaN,NaN,NaN,NaN,NaN
1926-05-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,2.483252,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# --- Compute start / end and subset data to the required window ---

fred = fred.loc[(desired_start_date_of_samples - relativedelta(months=3*HORIZON_IN_QUARTERS + 1)):last_date]
factors = factors.loc[(desired_start_date_of_samples - relativedelta(months=3*HORIZON_IN_QUARTERS)):last_date]
oap = oap.loc[(desired_start_date_of_samples - relativedelta(months=3*HORIZON_IN_QUARTERS)):last_date]
nfci = nfci.loc[(desired_start_date_of_samples - relativedelta(months=3*HORIZON_IN_QUARTERS)):last_date]

# --- Remove columns with missing values in the first training period (up to 1997-12-01) ---
fred_tr = fred.loc[:'1997-12-01'].dropna(axis=1)
factors_tr = factors.loc[:'1997-12-01'].dropna(axis=1)
oap_tr = oap.loc[:'1997-12-01'].dropna(axis=1)
fred_clean = fred[fred_tr.columns]
factors_clean = factors[factors_tr.columns]
oap_clean = oap[oap_tr.columns]


In [9]:
oap_clean

,AM,Accruals,AssetGrowth,BM,BMdec,Beta,BetaFP,BetaTailRisk,BidAskSpread,BookLeverage,...,VolSD,VolumeTrend,grcapx,grcapx3y,sinAlgo,std_turn,tang,zerotrade12M,zerotrade1M,zerotrade6M
date,,,,,,,,,,,,,,,,,,,,,
1960-01-01,2.865792,-1.496553,1.558198,3.409817,2.708306,-3.215553,-5.893109,-2.948154,4.089501,1.427640,...,4.169323,2.133555,-0.243489,-0.754139,1.309397,2.861723,-0.165619,5.656202,5.485962,5.970216
1960-02-01,-1.668649,4.425784,4.738621,1.622137,0.395568,-0.521860,1.440741,-1.382724,-1.657938,-0.930046,...,-1.474225,-2.090704,3.129878,2.707427,0.646189,-2.943003,-1.884503,-3.583121,-2.742413,-2.622438
1960-03-01,-4.033164,0.454437,-2.016133,-3.064029,-3.523729,-3.890434,-5.217810,-2.460037,-2.373326,1.476506,...,3.139626,1.225437,0.240340,0.204244,-0.687438,4.433302,1.937287,4.538267,4.779841,4.332288
1960-04-01,-1.500433,-2.007311,-1.083988,-1.218705,-1.533678,-5.276773,-5.888139,-4.920270,-2.900313,-0.668019,...,2.463723,-0.055441,-0.375405,-0.717202,0.433148,4.331450,-0.542935,3.655270,2.139517,3.256169
1960-05-01,-6.105031,-2.334311,-3.422076,-3.888498,-3.648621,-0.031533,2.098626,-1.498960,-2.827472,-0.644628,...,-2.088488,-1.398438,-1.665398,-2.262274,-2.038086,-3.530511,-2.118102,-5.496008,-7.529603,-7.257005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-01,-3.857898,-1.426554,-5.276677,-2.034590,-0.393782,-4.925949,-7.487172,-2.770608,-9.790082,-2.231970,...,3.388649,0.143575,1.440788,1.052862,-8.499483,8.725143,-1.299240,5.210274,2.293986,4.699576
2024-09-01,0.464554,-0.447110,1.326977,-0.502633,4.773794,1.234057,1.548505,0.017008,-1.787998,0.050628,...,-1.281481,0.026430,-0.181722,-1.842920,2.139061,0.360064,-2.051319,-0.957303,-1.170102,-1.365398
2024-10-01,-1.744559,-0.398807,-0.059058,0.259113,-1.675016,2.528316,-0.253378,2.387732,3.906344,-0.390145,...,-1.480556,-0.394200,-0.492340,1.021325,-2.986390,0.168005,6.024022,-3.374107,-1.350023,-3.630992


In [10]:
# Include lags for shelf models
fred_with_lags = fred_clean.copy()
for lag in range(1, 12):
    fred_with_lags = fred_with_lags.join(fred_clean.shift(lag).add_suffix(f'_t-{lag}'))
# fred_with_lags = fred_with_lags.iloc[12:]
fred_with_lags.head()

,RPI,W875RX1,DPCERA3M086SBEA,CMRMTSPLx,RETAILx,INDPRO,IPFPNSS,IPFINAL,IPCONGD,IPDCONGD,...,PCEPI_t-11,DDURRG3M086SBEA_t-11,DNDGRG3M086SBEA_t-11,DSERRG3M086SBEA_t-11,CES0600000008_t-11,CES2000000008_t-11,CES3000000008_t-11,DTCOLNVHFNM_t-11,DTCTHFNM_t-11,INVEST_t-11
sasdate,,,,,,,,,,,,,,,,,,,,,
1959-12-01,0.010192,0.011388,-0.001143,0.049614,-0.004644,0.059982,0.029466,0.023837,0.032241,0.120172,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1960-01-01,0.003226,0.004661,0.002791,0.016963,0.026606,0.025919,0.024097,0.029021,0.031234,0.103833,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1960-02-01,0.001147,0.000906,0.004361,0.014403,0.003696,-0.008937,-0.005686,-0.003436,-0.011454,-0.013856,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1960-03-01,0.001877,0.000905,0.014089,-0.028040,-0.001102,-0.009021,-0.003427,-0.001151,0.001151,-0.019967,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1960-04-01,0.003465,0.003612,0.015302,0.009853,0.025903,-0.007956,0.002286,0.001151,0.006882,-0.001188,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
financial_fred_vars = ['S&P 500', 'S&P div yield', 'S&P PE ratio', 'VIXCLSx', 'OILPRICEx', ]
fred_clean.columns

Index(['RPI', 'W875RX1', 'DPCERA3M086SBEA', 'CMRMTSPLx', 'RETAILx', 'INDPRO',
       'IPFPNSS', 'IPFINAL', 'IPCONGD', 'IPDCONGD',
       ...
       'PCEPI', 'DDURRG3M086SBEA', 'DNDGRG3M086SBEA', 'DSERRG3M086SBEA',
       'CES0600000008', 'CES2000000008', 'CES3000000008', 'DTCOLNVHFNM',
       'DTCTHFNM', 'INVEST'],
      dtype='object', length=116)

In [12]:
# --- Lag predictors ---
fred_with_lags_final = fred_with_lags.shift(3*HORIZON_IN_QUARTERS + 1)
fred_final = fred_clean.shift(3*HORIZON_IN_QUARTERS + 1)
factors_final = factors_clean.shift(3*HORIZON_IN_QUARTERS)
oap_final = oap_clean.shift(3*HORIZON_IN_QUARTERS)
nfci_final = nfci.shift(3*HORIZON_IN_QUARTERS)
fred_final_no_shift = fred_clean.copy()

# Subset data to final sample period
fred_with_lags_final = fred_with_lags_final.loc[desired_start_date_of_samples:last_date]
fred_final = fred_final.loc[desired_start_date_of_samples:last_date]
fred_final_no_shift = fred_final_no_shift.loc[desired_start_date_of_samples:last_date] # Indicies match to targets
factors_final = factors_final.loc[desired_start_date_of_samples:last_date]
oap_final = oap_final.loc[desired_start_date_of_samples:last_date]
nfci_final = nfci_final.loc[desired_start_date_of_samples:last_date]
targets = targets.loc[desired_start_date_of_samples:last_date]
ar1_predictors = ar1_predictors.loc[desired_start_date_of_samples:last_date]

fred_final_current = fred_clean.loc[desired_start_date_of_samples:last_date] # Not lagged, for current forecast

In [13]:
fred_with_lags_final = fred_with_lags_final.iloc[12:, :] # Adjust for lags

In [14]:
targets_first_date = targets.index[0].strftime('%Y-%m')
fred_first_date = fred_final.index[0].strftime('%Y-%m')
factors_first_date = factors_final.index[0].strftime('%Y-%m')
nfci_first_date = nfci_final.index[0].strftime('%Y-%m')

if not os.path.exists(os.path.join(DATA_DIR, 'MacroAtRisk/')):
    os.makedirs(os.path.join(DATA_DIR, 'MacroAtRisk/'))

# Append benchmark data to macro predictors
# vg_X = pd.read_csv(os.path.join(DATA_DIR, f'MacroAtRisk/us_vulnerable_growth_predictors_{HORIZON_IN_QUARTERS}q_{last_date_for_save}.csv'), index_col=0, parse_dates=True)
# ur_X = pd.read_csv(os.path.join(DATA_DIR, f'MacroAtRisk/us_unemployment_at_risk_predictors_{HORIZON_IN_QUARTERS}q_{last_date_for_save}.csv'), index_col=0, parse_dates=True)
# ir_X = pd.read_csv(os.path.join(DATA_DIR, f'MacroAtRisk/us_inflation_at_risk_predictors_{HORIZON_IN_QUARTERS}q_{last_date_for_save}.csv'), index_col=0, parse_dates=True)

# macro_with_bench = pd.concat([fred_final, vg_X, ur_X, ir_X], axis=1)

# macro_with_bench_first_date = macro_with_bench.index[0].strftime('%Y-%m')

print('Shape of macro predictors:', fred_final.shape)
print('Shape of financial predictors:', factors_final.shape)
print('Shape of NFCI predictors:', nfci_final.shape)
print('Shape of targets:', targets.shape)

targets.to_csv(os.path.join(DATA_DIR, f'MacroAtRisk/us_targets_{targets_first_date}--{last_date_for_save}.csv'))
fred_final.to_csv(os.path.join(DATA_DIR, f'MacroAtRisk/us_macro_predictors_{HORIZON_IN_QUARTERS}q_{fred_first_date}--{last_date_for_save}.csv'))
fred_with_lags_final.to_csv(os.path.join(DATA_DIR, f'MacroAtRisk/us_macro_predictors_with_12lags_{HORIZON_IN_QUARTERS}q_{fred_first_date}--{last_date_for_save}.csv'))

fred_final_current.to_csv(os.path.join(DATA_DIR, f'MacroAtRisk/us_macro_current_{last_date_for_save}.csv'))
# macro_with_bench.to_csv(os.path.join(DATA_DIR, f'MacroAtRisk/us_macro_with_benchmark_predictors_{HORIZON_IN_QUARTERS}q_{macro_with_bench_first_date}--{last_date_for_save}.csv'))
factors_final.to_csv(os.path.join(DATA_DIR, f'MacroAtRisk/us_financial_predictors_{HORIZON_IN_QUARTERS}q_{factors_first_date}--{last_date_for_save}.csv'))
oap_final.to_csv(os.path.join(DATA_DIR, f'MacroAtRisk/us_oap_financial_predictors_{HORIZON_IN_QUARTERS}q_{factors_first_date}--{last_date_for_save}.csv'))
nfci_final.to_csv(os.path.join(DATA_DIR, f'MacroAtRisk/us_nfci_{HORIZON_IN_QUARTERS}q_{nfci_first_date}--{last_date_for_save}.csv'))
ar1_predictors.to_csv(os.path.join(DATA_DIR, f'MacroAtRisk/us_ar1_predictors_{HORIZON_IN_QUARTERS}q_{last_date_for_save}.csv'))

# Also save factors by theme
factors_doc = pd.read_csv("/home/rproner/Documents/Data/JKP Documentation.csv")
themes = factors_doc['Theme'].unique()
for theme in themes:
    print(f'Saving {theme} factors')
    theme_factors = factors_doc.loc[factors_doc['Theme'] == theme, 'Variable'].values
    theme_factors = [f for f in theme_factors if f in factors_final.columns]
    factors_final.loc[:, theme_factors].to_csv(os.path.join(DATA_DIR, f'MacroAtRisk/us_financial_predictors_{theme.lower().replace(" ", "_")}_{HORIZON_IN_QUARTERS}q_{factors_first_date}--{last_date_for_save}.csv'))

print('Saved files:')
print(os.path.join(DATA_DIR, f'MacroAtRisk/us_targets_{last_date_for_save}.csv'))
print(os.path.join(DATA_DIR, f'MacroAtRisk/us_macro_predictors_{HORIZON_IN_QUARTERS}q_{last_date_for_save}.csv'))
# print(os.path.join(DATA_DIR, f'MacroAtRisk/us_macro_with_benchmark_predictors_{HORIZON_IN_QUARTERS}q_{last_date_for_save}.csv'))
print(os.path.join(DATA_DIR, f'MacroAtRisk/us_financial_predictors_{HORIZON_IN_QUARTERS}q_{last_date_for_save}.csv'))
print(os.path.join(DATA_DIR, f'MacroAtRisk/us_nfci_{HORIZON_IN_QUARTERS}q_{last_date_for_save}.csv'))


Shape of macro predictors: (768, 116)
Shape of financial predictors: (768, 136)
Shape of NFCI predictors: (647, 5)
Shape of targets: (768, 3)
Saving Accruals factors
Saving Debt Issuance factors
Saving Investment factors
Saving Low Leverage factors
Saving Low Risk factors
Saving Momentum factors
Saving Profitability factors
Saving Quality factors
Saving Seasonality factors
Saving Size factors
Saving Short-Term Reversal factors
Saving Value factors
Saved files:
/home/rproner/Documents/Data/MacroAtRisk/us_targets_2024-12.csv
/home/rproner/Documents/Data/MacroAtRisk/us_macro_predictors_4q_2024-12.csv
/home/rproner/Documents/Data/MacroAtRisk/us_financial_predictors_4q_2024-12.csv
/home/rproner/Documents/Data/MacroAtRisk/us_nfci_4q_2024-12.csv


In [15]:
print([t.lower().replace(" ", "_") for t in themes])

['accruals', 'debt_issuance', 'investment', 'low_leverage', 'low_risk', 'momentum', 'profitability', 'quality', 'seasonality', 'size', 'short-term_reversal', 'value']
